In [1]:
import pandas as pd

# EXPLORATION

In [2]:
df = pd.read_csv("ventes_boutique.csv")
df.head()

,id_commande,date_commande,client_id,ville_client,produit,categorie,quantite,prix_unitaire,note_satisfaction
0,CMD1240,2024-09-22,CL1021,Aneho,Sac a dos,Accessoire,3.0,71.52,4.0
1,CMD1060,2024-12-07,CL1083,Sokode,Jean,Bas,2.0,60.19,4.0
2,CMD1402,2024-09-10,CL1130,Sokode,Casquette,Accessoire,5.0,11.68,5.0
3,CMD1159,2024-03-20,CL1167,Sokode,Chaussettes,Bas,4.0,8.18,4.0
4,CMD1051,2024-03-20,CL1041,NaN,jean,Bas,4.0,47.38,3.0


In [3]:
df.shape

(490, 9)

In [4]:
df.dtypes

id_commande           object
date_commande         object
client_id             object
ville_client          object
produit               object
categorie             object
quantite             float64
prix_unitaire        float64
note_satisfaction    float64
dtype: object

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 490 entries, 0 to 489
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_commande        490 non-null    object 
 1   date_commande      490 non-null    object 
 2   client_id          490 non-null    object 
 3   ville_client       475 non-null    object 
 4   produit            490 non-null    object 
 5   categorie          490 non-null    object 
 6   quantite           482 non-null    float64
 7   prix_unitaire      475 non-null    float64
 8   note_satisfaction  445 non-null    float64
dtypes: float64(3), object(6)
memory usage: 34.6+ KB


In [6]:
df.isna().sum()

id_commande           0
date_commande         0
client_id             0
ville_client         15
produit               0
categorie             0
quantite              8
prix_unitaire        15
note_satisfaction    45
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(7)

# NETTOYAGE 

### Suppression des lignes dupliquées

In [8]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

### Nettoyez la colonne produit : supprimez les espaces inutiles et harmonisez la casse (par exemple tout mettre avec une majuscule initiale)

In [9]:
df['produit'] = df['produit'].str.strip()
df['produit'] = df['produit'].str.title()
display(df['produit'])

0        Sac A Dos
1             Jean
2        Casquette
3      Chaussettes
4             Jean
          ...     
485    Chaussettes
486          Veste
487        T-Shirt
488      Sac A Dos
489           Jean
Name: produit, Length: 483, dtype: object

### Conversion de la date

In [10]:
df["date_commande"] = pd.to_datetime(df["date_commande"], dayfirst=False, errors="coerce")

### Traitez les valeurs manquantes de prix_unitaire en les remplacant par la mediane du produit concerne 

In [11]:
df['prix_unitaire'] = df['prix_unitaire'].fillna(
    df.groupby('produit')['prix_unitaire'].transform('median')
)

In [12]:
df.isna().sum()

id_commande           0
date_commande        10
client_id             0
ville_client         15
produit               0
categorie             0
quantite              8
prix_unitaire         0
note_satisfaction    44
dtype: int64

### Traitez les valeurs manquantes de quantite en les remplacant par 1 (hypothese : une commande sans quantite renseignee correspond a 1 article)

In [13]:
df['quantite'] = df['quantite'].fillna(1)

In [14]:
df.isna().sum()

id_commande           0
date_commande        10
client_id             0
ville_client         15
produit               0
categorie             0
quantite              0
prix_unitaire         0
note_satisfaction    44
dtype: int64

### Traitez les valeurs manquantes de ville_client en les remplacant par la chaine "Inconnue"

In [15]:
df['ville_client'] = df['ville_client'].fillna("Inconnue")

In [16]:
df.isna().sum()

id_commande           0
date_commande        10
client_id             0
ville_client          0
produit               0
categorie             0
quantite              0
prix_unitaire         0
note_satisfaction    44
dtype: int64

###  Detectez les valeurs aberrantes

In [17]:
df.describe()

,date_commande,quantite,prix_unitaire,note_satisfaction
count,473,483.000000,483.000000,439.000000
mean,2024-07-11 23:14:20.042283520,2.345756,54.984679,3.630979
min,2024-01-01 00:00:00,1.000000,5.020000,1.000000
25%,2024-04-20 00:00:00,1.000000,17.870000,3.000000
50%,2024-07-11 00:00:00,2.000000,44.620000,4.000000
75%,2024-10-11 00:00:00,3.000000,68.010000,5.000000
max,2024-12-30 00:00:00,5.000000,2002.350000,5.000000
std,NaN,1.352160,102.944502,1.258958


In [18]:
outliers = df[df['prix_unitaire'] > 300]
display(outliers)

,id_commande,date_commande,client_id,ville_client,produit,categorie,quantite,prix_unitaire,note_satisfaction
115,CMD1450,2024-06-04,CL1062,Lome,Sac A Dos,Accessoire,2.0,457.20,5.0
376,CMD1054,2024-04-26,CL1006,Lome,Veste,Haut,5.0,2002.35,3.0
475,CMD1042,2024-06-27,CL1062,Lome,Jean,Bas,2.0,807.30,2.0


In [19]:
# Calcul de la médiane par produit
mediane_par_produit = df.groupby("produit")["prix_unitaire"].median()

# Remplacer les outliers par la médiane correspondante
df.loc[df["prix_unitaire"] > 300, "prix_unitaire"] = (
    df.loc[df["prix_unitaire"] > 300, "produit"].map(mediane_par_produit)
)

In [26]:
df[df['prix_unitaire'] > 300].value_counts()

Series([], Name: count, dtype: int64)

**La colonne note_satisfaction garde volontairement ses valeurs manquantes**